# NeuroVision AI - Kaggle Training Notebook
**Tema:** AI-Based Detection of Intracranial Aneurysms Using Deep Learning  
**Studente:** Blina Sopjani | ID: 69401 | Universum College  

---

## Pipeline
1. Setup & Imports
2. Dataset Loading (`/kaggle/input/`)
3. Train / Validation / Test Split (60/20/20)
4. CNN Baseline Training
5. ResNet-50 Transfer Learning
6. ResNet-101 Transfer Learning (Best)
7. Comparative Evaluation on Test Set
8. Save Checkpoints
9. RSNA Submission

**GPU:** P100 / T4 (Kaggle free)  
**Expected time:** ~8-10 hours

## 1. Setup & Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'pydicom', 'scikit-image', 'polars', '--quiet'], check=False)

import os, gc, json, time, warnings, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
INPUT_DIR   = Path('/kaggle/input/rsna-intracranial-aneurysm-detection')
WORKING_DIR = Path('/kaggle/working')
CKPT_DIR    = WORKING_DIR / 'checkpoints'
OUTPUT_DIR  = WORKING_DIR / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
    'Aneurysm Present',
]
N_CLASSES = len(LABEL_COLS)
ID_COL    = 'SeriesInstanceUID'

CONFIG = {
    'n_classes'    : N_CLASSES,
    'input_size'   : 224,
    'batch_size'   : 16,
    'learning_rate': 1e-4,
    'weight_decay' : 1e-5,
    'epochs'       : 50,
    'patience'     : 7,
    'pos_weight'   : 1.334,
    'freeze_epochs': 10,
    'dropout'      : 0.30,
    'random_seed'  : 42,
}

WINDOW_SETTINGS = {
    'CT'        : {'ww': 700,  'wl': 300},
    'CTA'       : {'ww': 700,  'wl': 300},
    'MR'        : {'ww': 3000, 'wl': 1500},
    'MRA'       : {'ww': 500,  'wl': 250},
    'MRI T1'    : {'ww': 3000, 'wl': 1500},
    'MRI T1post': {'ww': 3000, 'wl': 1500},
    'MRI T2'    : {'ww': 4000, 'wl': 2000},
    'DEFAULT'   : {'ww': 700,  'wl': 300},
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])
print('Config loaded OK')
print(f'  N_CLASSES  : {N_CLASSES}')
print(f'  pos_weight : {CONFIG["pos_weight"]}  (neg/pos = 2485/1863 from real data)')
print(f'  Batch size : {CONFIG["batch_size"]}')

## 2. Dataset Loading & Split (60/20/20)

In [ ]:
df = pd.read_csv(INPUT_DIR / 'train.csv')

print(f'Dataset loaded: {len(df):,} cases')
print(f'\nClass distribution:')
print(df['Aneurysm Present'].value_counts())
print(f'\nModality distribution:')
print(df['Modality'].value_counts())
print(f'\nMissing values:')
mv = df.isnull().sum()
print(mv[mv > 0] if mv.sum() > 0 else 'None')

# Fill missing values
df['PatientAge'] = df['PatientAge'].fillna(df['PatientAge'].median())
df['PatientSex'] = df['PatientSex'].fillna('Unknown')
print('\nMissing values filled OK')

In [ ]:
# Split 60/20/20 with stratification on 'Aneurysm Present'
df_train, df_temp = train_test_split(
    df,
    test_size    = 0.40,
    stratify     = df['Aneurysm Present'],
    random_state = CONFIG['random_seed'],
)
df_val, df_test = train_test_split(
    df_temp,
    test_size    = 0.50,
    stratify     = df_temp['Aneurysm Present'],
    random_state = CONFIG['random_seed'],
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Split results (seed={CONFIG["random_seed"]}):')
print(f'{"Split":<12} {"N":>6} {"% Total":>9} {"Positive%":>11}')
print('-' * 42)
for name, subset in [('Train', df_train), ('Validation', df_val), ('Test', df_test)]:
    pos_pct = subset['Aneurysm Present'].mean() * 100
    print(f'{name:<12} {len(subset):>6,} {len(subset)/len(df)*100:>8.1f}%  {pos_pct:>9.1f}%')

df_test[[ID_COL, 'Aneurysm Present']].to_csv(OUTPUT_DIR / 'test_split_ids.csv', index=False)
print('\nTest IDs saved to outputs/')

## 3. Dataset Class & Preprocessing

In [ ]:
import pydicom
from skimage.transform import resize as sk_resize

def apply_window(arr: np.ndarray, modality: str) -> np.ndarray:
    s  = WINDOW_SETTINGS.get(modality, WINDOW_SETTINGS['DEFAULT'])
    lo = s['wl'] - s['ww'] / 2
    hi = s['wl'] + s['ww'] / 2
    return ((np.clip(arr, lo, hi) - lo) / (hi - lo + 1e-8)).astype(np.float32)


def load_series_image(series_id: str, modality: str) -> np.ndarray:
    """
    Load DICOM series -> [3, 224, 224] tensor.
    Selects 3 central slices (Q25, Q50, Q75) as 3 channels.
    Applies windowing, resize, ImageNet normalization.
    """
    series_path = INPUT_DIR / 'series' / series_id
    dcm_files   = sorted(series_path.glob('*.dcm'))

    if not dcm_files:
        return np.zeros((3, 224, 224), dtype=np.float32)

    slices = []
    for dcm_path in dcm_files:
        try:
            ds  = pydicom.dcmread(str(dcm_path), force=True)
            arr = ds.pixel_array.astype(np.float32)
            slope     = float(getattr(ds, 'RescaleSlope',     1.0))
            intercept = float(getattr(ds, 'RescaleIntercept', 0.0))
            hu  = arr * slope + intercept
            ipp = getattr(ds, 'ImagePositionPatient', [0, 0, 0])
            z   = float(ipp[2]) if hasattr(ipp, '__len__') else 0.0
            slices.append({'z': z, 'hu': hu})
        except Exception:
            continue

    if not slices:
        return np.zeros((3, 224, 224), dtype=np.float32)

    slices.sort(key=lambda s: s['z'])
    total   = len(slices)
    indices = [total // 4, total // 2, 3 * total // 4]
    indices = [min(i, total - 1) for i in indices]

    channels = [apply_window(slices[i]['hu'], modality) for i in indices]
    arr_3ch  = np.stack(channels, axis=0)  # [3, H, W]

    arr_3ch = np.stack([
        sk_resize(arr_3ch[i], (224, 224), anti_aliasing=True, preserve_range=True)
        for i in range(3)
    ]).astype(np.float32)

    for i in range(3):
        arr_3ch[i] = (arr_3ch[i] - IMAGENET_MEAN[i]) / (IMAGENET_STD[i] + 1e-8)

    return arr_3ch


class AneurysmDataset(Dataset):
    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        series_id = row[ID_COL]
        modality  = str(row.get('Modality', 'CTA'))
        labels    = torch.tensor(
            row[LABEL_COLS].values.astype(np.float32), dtype=torch.float32
        )
        image = load_series_image(series_id, modality)
        image = torch.tensor(image, dtype=torch.float32)

        if self.is_train:
            if torch.rand(1) > 0.5:
                image = torch.flip(image, dims=[2])
            if torch.rand(1) > 0.6:
                import torchvision.transforms.functional as TF
                angle = float(torch.empty(1).uniform_(-15, 15))
                image = TF.rotate(image, angle)
        return image, labels, series_id


# Test quick sample
sample_id  = df_train[ID_COL].iloc[0]
sample_mod = df_train['Modality'].iloc[0]
img = load_series_image(sample_id, sample_mod)
print(f'Sample image shape : {img.shape}')
print(f'Min/Max values     : [{img.min():.3f}, {img.max():.3f}]')
print(f'Modality           : {sample_mod}')

In [ ]:
ds_train = AneurysmDataset(df_train, is_train=True)
ds_val   = AneurysmDataset(df_val,   is_train=False)
ds_test  = AneurysmDataset(df_test,  is_train=False)

loader_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=4, pin_memory=True)
loader_val   = DataLoader(ds_val,   batch_size=CONFIG['batch_size']*2,
                          shuffle=False, num_workers=4, pin_memory=True)
loader_test  = DataLoader(ds_test,  batch_size=CONFIG['batch_size']*2,
                          shuffle=False, num_workers=4, pin_memory=True)

print(f'DataLoaders ready')
print(f'  Train batches : {len(loader_train):,}')
print(f'  Val batches   : {len(loader_val):,}')
print(f'  Test batches  : {len(loader_test):,}')

## 4. Model Architectures

In [ ]:
# ── CNN BASELINE ──────────────────────────────────────────────────────────
class CNNBaseline(nn.Module):
    """5 conv blocks from scratch. ~6.2M params. Reference model."""
    def __init__(self, n_classes=14, dropout=0.30):
        super().__init__()
        def conv_block(in_ch, out_ch, drop=0.15):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2), nn.Dropout2d(p=drop),
            )
        self.features = nn.Sequential(
            conv_block(3,   32,  0.10), conv_block(32,  64,  0.10),
            conv_block(64,  128, 0.15), conv_block(128, 256, 0.15),
            conv_block(256, 512, 0.20),
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return torch.sigmoid(self.classifier(self.pool(self.features(x))))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── RESNET (50/101) ────────────────────────────────────────────────────────
class ResNetModel(nn.Module):
    """
    ResNet-50 or ResNet-101 with ImageNet Transfer Learning.
    14-output sigmoid head. Transfer learning schedule:
      Epochs 1-10  : backbone frozen
      Epoch 11-50  : Layer4 + head fine-tuned
    """
    def __init__(self, variant='resnet101', n_classes=14, dropout=0.30, pretrained=True):
        super().__init__()
        self.variant = variant
        if variant == 'resnet50':
            weights  = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            backbone = models.resnet50(weights=weights)
        else:
            weights  = models.ResNet101_Weights.IMAGENET1K_V2 if pretrained else None
            backbone = models.resnet101(weights=weights)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(), nn.Linear(2048, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(512, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.67), nn.Linear(128, n_classes),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_layer4(self):
        for p in list(self.backbone.children())[-1].parameters():
            p.requires_grad = True
    def forward(self, x):
        return torch.sigmoid(self.head(self.pool(self.backbone(x))))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── LOSS & METRIC ──────────────────────────────────────────────────────────
class WeightedBCELoss(nn.Module):
    def __init__(self, pos_weight=1.334):
        super().__init__()
        self.pos_weight = pos_weight
    def forward(self, preds, targets):
        pw     = torch.tensor(self.pos_weight, device=preds.device)
        logits = torch.log(preds.clamp(1e-6) / (1 - preds).clamp(1e-6))
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)


def rsna_weighted_auc(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """RSNA score = 0.5*AUC(Aneurysm Present) + 0.5*mean(AUC(13 locations))"""
    per_auc = {}
    for i, label in enumerate(LABEL_COLS):
        if len(np.unique(y_true[:, i])) > 1:
            per_auc[label] = roc_auc_score(y_true[:, i], y_pred[:, i])
        else:
            per_auc[label] = 0.5
    vals         = list(per_auc.values())
    aneurysm_auc = vals[-1]
    loc_mean     = float(np.mean(vals[:-1]))
    final        = 0.5 * aneurysm_auc + 0.5 * loc_mean
    return {'final': round(final,4), 'aneurysm': round(aneurysm_auc,4),
            'loc_mean': round(loc_mean,4), 'per_label': per_auc}


# Info
m1 = CNNBaseline(N_CLASSES)
m2 = ResNetModel('resnet50',  N_CLASSES, pretrained=False)
m3 = ResNetModel('resnet101', N_CLASSES, pretrained=False)
print(f'CNN Baseline : {m1.count_parameters():>12,} params')
print(f'ResNet-50    : {m2.count_parameters():>12,} params')
print(f'ResNet-101   : {m3.count_parameters():>12,} params')
del m1, m2, m3
gc.collect()

## 5. Training Loop

In [ ]:
def train_model(model, model_name: str, config: dict,
                loader_train, loader_val) -> tuple:
    """
    Full training loop:
    - WeightedBCE Loss + AdamW + Cosine LR scheduler
    - Early stopping (patience=7)
    - Transfer learning schedule (freeze->unfreeze Layer4)
    - RSNA Weighted AUC tracking
    - Best checkpoint saving
    """
    model     = model.to(DEVICE)
    criterion = WeightedBCELoss(pos_weight=config['pos_weight'])
    is_resnet = isinstance(model, ResNetModel)

    if is_resnet:
        model.freeze_backbone()
        print(f'  Backbone FROZEN for epochs 1-{config["freeze_epochs"]}')

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['learning_rate'], weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs'], eta_min=config['learning_rate'] * 0.01
    )

    history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[], 'lr':[]}
    best_auc, best_epoch, patience_cnt = 0.0, 0, 0
    ckpt_path = CKPT_DIR / f'{model_name}_best.pt'

    print(f'\n{"="*60}')
    print(f'  TRAINING: {model_name}')
    print(f'{"="*60}')

    for epoch in range(1, config['epochs'] + 1):
        t0 = time.time()

        if is_resnet and epoch == config['freeze_epochs'] + 1:
            model.unfreeze_layer4()
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=config['learning_rate'] * 0.1,
                weight_decay=config['weight_decay']
            )
            print(f'  -> Epoch {epoch}: Layer4 UNFROZEN - fine-tuning active')

        # TRAIN
        model.train()
        tr_loss, tr_preds, tr_targets = 0.0, [], []
        for images, labels, _ in loader_train:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item()
            tr_preds.append(out.detach().cpu().numpy())
            tr_targets.append(labels.detach().cpu().numpy())

        tr_p = np.vstack(tr_preds)
        tr_t = np.vstack(tr_targets)
        try:   tr_auc = roc_auc_score(tr_t, tr_p, average='macro')
        except: tr_auc = 0.5

        # VALIDATE
        model.eval()
        vl_loss, vl_preds, vl_targets = 0.0, [], []
        with torch.no_grad():
            for images, labels, _ in loader_val:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                out  = model(images)
                loss = criterion(out, labels)
                vl_loss += loss.item()
                vl_preds.append(out.cpu().numpy())
                vl_targets.append(labels.cpu().numpy())

        vl_p    = np.vstack(vl_preds)
        vl_t    = np.vstack(vl_targets)
        metrics = rsna_weighted_auc(vl_t, vl_p)
        vl_auc  = metrics['final']

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(round(tr_loss / len(loader_train), 4))
        history['val_loss'].append(round(vl_loss / len(loader_val), 4))
        history['train_auc'].append(round(float(tr_auc), 4))
        history['val_auc'].append(round(vl_auc, 4))
        history['lr'].append(round(lr, 8))

        elapsed = time.time() - t0
        flag    = ''
        if vl_auc > best_auc:
            best_auc, best_epoch, patience_cnt = vl_auc, epoch, 0
            torch.save({'epoch': epoch, 'val_auc': vl_auc,
                        'model_state': model.state_dict(), 'config': config}, ckpt_path)
            flag = '  * BEST'
        else:
            patience_cnt += 1

        print(f'  Ep {epoch:3d}/{config["epochs"]} | '
              f'Loss {tr_loss/len(loader_train):.4f}/{vl_loss/len(loader_val):.4f} | '
              f'AUC {tr_auc:.4f}/{vl_auc:.4f} | '
              f'RSNA {metrics["final"]:.4f} | '
              f'LR {lr:.1e} | {elapsed:.0f}s{flag}')

        if patience_cnt >= config['patience']:
            print(f'\n  Early stopping at epoch {epoch} (best: {best_epoch}, AUC={best_auc:.4f})')
            break

        del images, labels, out
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    with open(OUTPUT_DIR / f'{model_name}_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\n  {model_name} DONE')
    print(f'  Best Epoch : {best_epoch}')
    print(f'  Best AUC   : {best_auc:.4f}')
    print(f'  Checkpoint : {ckpt_path}')
    return history, best_auc, str(ckpt_path)


print('Training function ready')

## 6. Train All Models

In [ ]:
print('Training CNN Baseline...')
cnn_model = CNNBaseline(n_classes=N_CLASSES, dropout=CONFIG['dropout'])
cnn_history, cnn_best_auc, _ = train_model(
    cnn_model, 'CNN_baseline', CONFIG, loader_train, loader_val
)
del cnn_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
print('Training ResNet-50...')
r50_model = ResNetModel('resnet50', n_classes=N_CLASSES,
                        dropout=CONFIG['dropout'], pretrained=True)
r50_history, r50_best_auc, _ = train_model(
    r50_model, 'ResNet50', CONFIG, loader_train, loader_val
)
del r50_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
print('Training ResNet-101 (best model)...')
r101_model = ResNetModel('resnet101', n_classes=N_CLASSES,
                         dropout=CONFIG['dropout'], pretrained=True)
r101_history, r101_best_auc, _ = train_model(
    r101_model, 'ResNet101', CONFIG, loader_train, loader_val
)
del r101_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 7. Evaluate on Test Set

In [ ]:
def evaluate_on_test(model_name: str, ckpt_path, is_resnet=True) -> dict:
    if not is_resnet:
        model = CNNBaseline(n_classes=N_CLASSES)
    else:
        variant = 'resnet50' if '50' in model_name else 'resnet101'
        model   = ResNetModel(variant, n_classes=N_CLASSES, pretrained=False)

    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval().to(DEVICE)
    print(f'  Checkpoint loaded: epoch={ckpt["epoch"]}, val_auc={ckpt["val_auc"]:.4f}')

    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, labels, _ in loader_test:
            out = model(images.to(DEVICE))
            all_preds.append(out.cpu().numpy())
            all_targets.append(labels.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    rsna   = rsna_weighted_auc(y_true, y_pred)

    # Bootstrap 95% CI
    np.random.seed(42)
    boot_aucs = []
    for _ in range(500):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        aucs_b = []
        for i in range(N_CLASSES):
            if len(np.unique(y_true[idx, i])) > 1:
                aucs_b.append(roc_auc_score(y_true[idx, i], y_pred[idx, i]))
        if aucs_b:
            boot_aucs.append(0.5 * aucs_b[-1] + 0.5 * np.mean(aucs_b[:-1]))
    ci_low  = round(float(np.percentile(boot_aucs, 2.5)), 4)
    ci_high = round(float(np.percentile(boot_aucs, 97.5)), 4)

    # Binary metrics on Aneurysm Present (index 13)
    y_tb = y_true[:, -1].astype(int)
    y_pb = (y_pred[:, -1] >= 0.5).astype(int)
    tp = int(((y_tb==1)&(y_pb==1)).sum())
    tn = int(((y_tb==0)&(y_pb==0)).sum())
    fp = int(((y_tb==0)&(y_pb==1)).sum())
    fn = int(((y_tb==1)&(y_pb==0)).sum())

    sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    ppv  = tp/(tp+fp) if (tp+fp)>0 else 0.0
    npv  = tn/(tn+fn) if (tn+fn)>0 else 0.0
    f1   = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0.0
    acc  = (tp+tn)/(tp+tn+fp+fn)

    del model
    gc.collect()

    return {
        'model': model_name,
        'rsna_weighted_auc': rsna['final'],
        'aneurysm_auc':      rsna['aneurysm'],
        'location_auc':      rsna['loc_mean'],
        'auc_ci_low':        ci_low,
        'auc_ci_high':       ci_high,
        'accuracy':          round(acc,  4),
        'sensitivity':       round(sens, 4),
        'specificity':       round(spec, 4),
        'ppv':               round(ppv,  4),
        'npv':               round(npv,  4),
        'f1_score':          round(f1,   4),
        'confusion_matrix':  {'TP':tp,'TN':tn,'FP':fp,'FN':fn},
        'per_label_auc':     {k: round(v,4) for k,v in rsna['per_label'].items()},
        'n_test':            len(y_true),
    }


print('Evaluating all models on test set...')
results_cnn  = evaluate_on_test('CNN Baseline', CKPT_DIR/'CNN_baseline_best.pt',  is_resnet=False)
results_r50  = evaluate_on_test('ResNet-50',    CKPT_DIR/'ResNet50_best.pt',      is_resnet=True)
results_r101 = evaluate_on_test('ResNet-101',   CKPT_DIR/'ResNet101_best.pt',     is_resnet=True)
all_results  = [results_cnn, results_r50, results_r101]

# Print comparison table
print(f'\n{"="*70}')
print(f'  COMPARISON TABLE - TEST SET (N={results_cnn["n_test"]})')
print(f'{"="*70}')
print(f'{"Metric":<26} {"CNN Baseline":>14} {"ResNet-50":>12} {"ResNet-101":>12}')
print('-'*66)
for label, key in [
    ('RSNA Weighted AUC',  'rsna_weighted_auc'),
    ('Aneurysm AUC',       'aneurysm_auc'),
    ('Location Mean AUC',  'location_auc'),
    ('Accuracy',           'accuracy'),
    ('Sensitivity',        'sensitivity'),
    ('Specificity',        'specificity'),
    ('Precision (PPV)',    'ppv'),
    ('F1-Score',           'f1_score'),
]:
    vals = [r[key] for r in all_results]
    best = max(vals)
    row  = [f'{v:.4f}{" *" if v==best else "  "}' for v in vals]
    print(f'{label:<26} {row[0]:>14} {row[1]:>12} {row[2]:>12}')
print('-'*66)
for r in all_results:
    print(f'  {r["model"]:<14} 95% CI: [{r["auc_ci_low"]:.3f} - {r["auc_ci_high"]:.3f}]')

diff = results_r101['rsna_weighted_auc'] - results_cnn['rsna_weighted_auc']
print(f'\n  H1: ResNet-101 ({results_r101["rsna_weighted_auc"]:.4f}) > CNN ({results_cnn["rsna_weighted_auc"]:.4f})  delta={diff:+.4f}')
print(f'  -> {"VERIFIED" if diff > 0 else "NOT VERIFIED"}')

In [ ]:
output_data = {
    'split_info': {
        'train': len(df_train), 'val': len(df_val),
        'test':  len(df_test),  'seed': CONFIG['random_seed'],
        'ratio': '60/20/20',
    },
    'model_results': all_results,
    'training_history': {
        'CNN_baseline': cnn_history,
        'ResNet50':     r50_history,
        'ResNet101':    r101_history,
    },
}
with open(OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)

rows = [{
    'Model':        r['model'],
    'RSNA_AUC':     r['rsna_weighted_auc'],
    'Aneurysm_AUC': r['aneurysm_auc'],
    'Location_AUC': r['location_auc'],
    'CI_Low':       r['auc_ci_low'],
    'CI_High':      r['auc_ci_high'],
    'Accuracy':     r['accuracy'],
    'Sensitivity':  r['sensitivity'],
    'Specificity':  r['specificity'],
    'PPV':          r['ppv'],
    'NPV':          r['npv'],
    'F1_Score':     r['f1_score'],
    'N_Test':       r['n_test'],
} for r in all_results]
pd.DataFrame(rows).to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

print('Results saved:')
print(f'  {OUTPUT_DIR}/evaluation_results.json')
print(f'  {OUTPUT_DIR}/model_comparison.csv')
print('\nCheckpoints:')
for f in sorted(CKPT_DIR.glob('*.pt')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## 8. RSNA Submission

In [ ]:
import polars as pl
import kaggle_evaluation.rsna_inference_server

_sub_model = ResNetModel('resnet101', n_classes=N_CLASSES, pretrained=False)
_sub_ckpt  = torch.load(CKPT_DIR / 'ResNet101_best.pt', map_location=DEVICE)
_sub_model.load_state_dict(_sub_ckpt['model_state'])
_sub_model.eval().to(DEVICE)
print(f'Submission model: ResNet-101 | val_auc={_sub_ckpt["val_auc"]:.4f}')


def predict(series_path: str):
    series_id = os.path.basename(series_path.rstrip('/\\'))
    mod_row   = df[df[ID_COL] == series_id]
    modality  = str(mod_row['Modality'].iloc[0]) if len(mod_row) > 0 else 'CTA'
    image     = load_series_image(series_id, modality)
    with torch.no_grad():
        tensor = torch.tensor(image).unsqueeze(0).to(DEVICE)
        probs  = _sub_model(tensor).squeeze(0).cpu().numpy().tolist()
    predictions = pl.DataFrame(
        data=[[series_id] + probs], schema=[ID_COL] + LABEL_COLS, orient='row'
    )
    shutil.rmtree('/kaggle/shared', ignore_errors=True)
    return predictions.drop(ID_COL)


inference_server = kaggle_evaluation.rsna_inference_server.RSNAInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway()
    submission = pl.read_parquet('/kaggle/working/submission.parquet')
    print(f'Submission shape: {submission.shape}')
    print(submission.head(3))

## 9. After Execution - Download Results

After the notebook finishes (~8-10 hours), click **Output -> Download All**:

- `checkpoints/CNN_baseline_best.pt` (~25 MB)
- `checkpoints/ResNet50_best.pt` (~100 MB)
- `checkpoints/ResNet101_best.pt` (~170 MB)
- `outputs/evaluation_results.json`
- `outputs/model_comparison.csv`

Place them in your local repo:
```
RSNA-Aneurysm/
├── checkpoints/   <- .pt files here
└── outputs/       <- .json and .csv files here
```

---
**Student:** Blina Sopjani | **ID:** 69401 | **Universum College